In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("aielawady/arsl-256")

print("Path to dataset files:", path)

100%|██████████| 72.5M/72.5M [00:04<00:00, 18.1MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/aielawady/arsl-256/versions/1


In [2]:
import os
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from torchvision.datasets.folder import default_loader

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define path to dataset
data_dir = path  # path from kagglehub

# Image transformations for VGG16 (no augmentation yet)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

# Load dataset using ImageFolder
full_dataset = ImageFolder(root=data_dir, transform=transform)

# Split into train and validation sets (80/20)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

# DataLoaders
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print("Dataloader setup complete.")


Using device: cuda
Dataloader setup complete.


In [3]:
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

# Define VGG16 model (no BatchNorm)
class VGG16(nn.Module):
    def __init__(self, num_classes):
        super(VGG16, self).__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Block 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Block 3
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Block 4
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Block 5
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

# Instantiate model
num_classes = len(full_dataset.classes)
model = VGG16(num_classes).to(device)

# Loss & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Training function
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10):
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}]")
        for images, labels in loop:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            loop.set_postfix(loss=loss.item(), acc=100 * correct / total)

        # Validation accuracy
        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_acc = 100 * val_correct / val_total
        print(f"Validation Accuracy: {val_acc:.2f}%\n")

# Train
train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10)
# from https://medium.com/@mygreatlearning/everything-you-need-to-know-about-vgg16-7315defb5918

Epoch [1/10]: 100%|██████████| 197/197 [01:53<00:00,  1.74it/s, acc=3.64, loss=3.48]


Validation Accuracy: 3.50%



Epoch [2/10]: 100%|██████████| 197/197 [01:53<00:00,  1.74it/s, acc=4.06, loss=3.45]


Validation Accuracy: 4.26%



Epoch [3/10]: 100%|██████████| 197/197 [01:53<00:00,  1.74it/s, acc=3.71, loss=3.43]


...

Validation Accuracy: 3.50%



Epoch [9/10]: 100%|██████████| 197/197 [01:52<00:00,  1.75it/s, acc=3.87, loss=3.46]


Validation Accuracy: 3.50%



Epoch [10/10]: 100%|██████████| 197/197 [01:52<00:00,  1.75it/s, acc=4.07, loss=3.42]


Validation Accuracy: 3.50%



In [4]:
class VGG16_BN(nn.Module):
    def __init__(self, num_classes):
        super(VGG16_BN, self).__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            # Block 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            # Block 3
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            # Block 4
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            # Block 5
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

model_bn = VGG16_BN(num_classes).to(device)
optimizer_bn = optim.Adam(model_bn.parameters(), lr=0.0001)

In [5]:
model_bn = VGG16_BN(num_classes).to(device)
optimizer_bn = optim.Adam(model_bn.parameters(), lr=0.0001)

train_model(model_bn, train_loader, val_loader, criterion, optimizer_bn, num_epochs=10)


Epoch [1/10]: 100%|██████████| 197/197 [02:11<00:00,  1.50it/s, acc=2.96, loss=3.46]


Validation Accuracy: 3.31%



Epoch [2/10]: 100%|██████████| 197/197 [02:11<00:00,  1.49it/s, acc=3.69, loss=3.46]


Validation Accuracy: 3.88%



Epoch [3/10]: 100%|██████████| 197/197 [02:11<00:00,  1.50it/s, acc=3.71, loss=3.28]


...

Validation Accuracy: 62.51%



Epoch [9/10]: 100%|██████████| 197/197 [02:11<00:00,  1.50it/s, acc=62.7, loss=2.46]


Validation Accuracy: 70.85%



Epoch [10/10]: 100%|██████████| 197/197 [02:11<00:00,  1.50it/s, acc=70, loss=0.963]


Validation Accuracy: 67.79%



In [6]:
# Data augmentation for training
train_transform_augmented = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
])

# Keep validation transform clean
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

# Reload datasets with transforms
train_dataset_aug = ImageFolder(root=data_dir, transform=train_transform_augmented)
val_dataset_clean = ImageFolder(root=data_dir, transform=val_transform)

# Use same random split indices as before
train_indices, val_indices = torch.utils.data.random_split(
    list(range(len(train_dataset_aug))),
    [int(0.8 * len(train_dataset_aug)), len(train_dataset_aug) - int(0.8 * len(train_dataset_aug))]
)

train_dataset_aug = torch.utils.data.Subset(train_dataset_aug, train_indices)
val_dataset_clean = torch.utils.data.Subset(val_dataset_clean, val_indices)

# DataLoaders
train_loader_aug = DataLoader(train_dataset_aug, batch_size=batch_size, shuffle=True)
val_loader_clean = DataLoader(val_dataset_clean, batch_size=batch_size, shuffle=False)

print("Augmented dataloaders ready.")


Augmented dataloaders ready.


In [8]:
# New model for fair comparison
model_aug = VGG16_BN(num_classes).to(device)
optimizer_aug = optim.Adam(model_aug.parameters(), lr=0.0001)

train_model(model_aug, train_loader_aug, val_loader_clean, criterion, optimizer_aug, num_epochs=10)


Epoch [1/10]: 100%|██████████| 197/197 [02:14<00:00,  1.46it/s, acc=3.39, loss=3.48]


Validation Accuracy: 3.69%



Epoch [2/10]: 100%|██████████| 197/197 [02:12<00:00,  1.48it/s, acc=3.28, loss=3.38]


Validation Accuracy: 4.14%



Epoch [3/10]: 100%|██████████| 197/197 [02:12<00:00,  1.48it/s, acc=3.52, loss=3.46]


...

Validation Accuracy: 17.25%



Epoch [9/10]: 100%|██████████| 197/197 [02:13<00:00,  1.48it/s, acc=43.5, loss=2.11]


Validation Accuracy: 9.04%



Epoch [10/10]: 100%|██████████| 197/197 [02:12<00:00,  1.48it/s, acc=50.6, loss=1.3]


Validation Accuracy: 29.85%



In [9]:


# The code trains three VGG16 models on the ARSL-256 dataset:
# 1. VGG16 without Batch Normalization: doesn't learn quick enough and has validation accuracy of 3.5% after 10 epoches
# 2. VGG16 with Batch Normalization: Shows improved validation accuracy, around 70% after 10 epochs showing huge improvment in learning time
# 3. VGG16 with Batch Normalization and data augmentation: Validation accuracy should be comparable to or slightly better than the previous model,
# demonstrating the effect of augmentation. but after more epoches

